> ## SOLUTION / ANSWER KEY &mdash; Lab 9.1
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-01-ground-a-figure.ipynb`](../lab-01-ground-a-figure.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 9.1 &mdash; Ground a Figure (Extract + Cite)

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 5 &middot; Module 9 &mdash; Agents in Finance, Healthcare &amp; Cybersecurity**

### What you'll do
- Extract a reported figure together with its source
- Check a figure is grounded -- it carries a citation
- See why a figure with no source is unusable here

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it** (the real grounding / citation / compute logic, or the real `create_agent` wiring), then **Run it** and read the output &mdash; and, for the agent labs, the real **message trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working, grounded insight agent you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`). The insight agent is a **real** `create_agent` driven by a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")`, key in `.env` as `GROQ_API_KEY`). You are building the **financial-report insight agent** &mdash; the client's Lab 5.1. It is **informational only**: it grounds &amp; **cites** every figure, gives **no advice**, and has **no trade tool** (`place_trade` is defined but never bound &mdash; a human analyst decides). A `@tool` must **catch its own errors and return a string** &mdash; a tool that raises can abort the whole run. If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing.

**Reference:** [Module 9 slides &mdash; The domain-agent pattern](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 9 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (the Day-5 provider)

WORK = "/tmp/biaa-lab-09-01"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is set. Day-5 labs call a REAL hosted model (Groq)."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-5 provider: a REAL hosted model. openai/gpt-oss-20b is a reliable tool-caller via create_agent.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY set | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
In a high-stakes domain the model's memory isn't just unreliable &mdash; it's **dangerous** (deck
slides 4&ndash;5, 14). Every figure must be **grounded**: pulled from the actual document, carrying its
**source** so a human can verify it. A number without a citation is a liability dressed up as an answer.
Here you build the core move &mdash; extract a figure **and** its source &mdash; the discipline the real
insight agent (Lab 11) enforces with a read-only `extract_figure` tool.

In [ ]:
# Memory vs the document -- why this matters.
# Ask a model for revenue FROM MEMORY and it may answer a plausible-but-wrong number, with no source.
MEMORY_GUESS = 118.0            # what a model might "recall" -- plausible, and WRONG
GROUNDED = {"value": 120.0, "source": "p4, income stmt"}
print("from memory (ungrounded):", MEMORY_GUESS, "M   <- no source, do NOT trust")
print("grounded (from filing)  :", GROUNDED["value"], "M  [", GROUNDED["source"], "]")
print("gap:", round(GROUNDED["value"] - MEMORY_GUESS, 1), "M of pure hallucination risk")

## Build it
Complete `extract_figure` (return the figure with its source) and `is_grounded` (it must carry a
citation), then run the cell to watch a grounded vs an ungrounded figure.

In [ ]:
# A small financial report -- each figure carries its SOURCE, so every claim can be cited.
REPORT = {
    "revenue":    {"value": 120.0, "unit": "M", "source": "p4, income stmt"},
    "net_income": {"value": 9.0,   "unit": "M", "source": "p4, income stmt"},
    "total_debt": {"value": 40.0,  "unit": "M", "source": "p7, balance sheet"},
}
PRIOR = {"revenue": 107.1, "net_income": 9.7, "total_debt": 25.0}   # prior period, for YoY

def extract_figure(name, report):
    # return the figure dict {value, source} from the report, or None if it is not there
    return report.get(name)

def is_grounded(fig):
    # grounded = the figure exists AND carries a non-empty source citation
    return fig is not None and bool(fig.get("source"))

try:
    rev = extract_figure("revenue", REPORT)
    print("revenue :", rev)
    print("grounded:", is_grounded(rev))
    print("missing :", extract_figure("ebitda", REPORT))
    print("no-source grounded?", is_grounded({"value": 5.0}))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- A grounded figure carries **`source`** &mdash; the page cite a human can check. A bare `{"value": ...}` is **not** grounded.
- A missing figure returns **`None`** rather than a guess &mdash; the agent must say "not in the filing", never invent.
- This is exactly what the real `extract_figure` tool does in Lab 11: pull the value **with** its source.

## Your turn (open task &mdash; no grader)
Add a new figure to `REPORT` (say `"opex"` with a source), extract it, and confirm `is_grounded` is
True; then add one **without** a source and confirm it's False. **What good looks like:** every figure the
agent will state carries a page cite, and anything uncited is caught before it ships.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
REPORT["opex"]    = {"value": 30.0, "unit": "M", "source": "p5, income stmt"}
REPORT["mystery"] = {"value": 7.0, "unit": "M"}   # deliberately NO source
print("opex grounded?   ", is_grounded(extract_figure("opex", REPORT)))     # True  -- carries a page cite
print("mystery grounded?", is_grounded(extract_figure("mystery", REPORT)))  # False -- uncited, caught before it ships

---
### Key takeaway
Ground every figure: extract it WITH its source. A number without a citation can't be verified -- and in finance, health or cyber, an unverifiable claim doesn't ship.

[&#8592; All Module 9 labs](./index.html) &nbsp;&middot;&nbsp; [Module 9 slides](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>